# 05 - Data and model analysis

- **Part A: the data.** Solubility per solvent, overlaps (Venn), solvents that behave alike, effect of RT and molecular weight, composition of the training set.
- **Part B: the model.** Analyses that go beyond the overall score of notebook 04: per-solvent scores, confusion matrices, learning curve, calibration, top-k, feature importance, error analysis and effect of each plate.

Re-run after each new plate, once `scripts/data_prep/add_new_molecules.py` has been run. Part B takes a few minutes.

## Part A: the data

This part describes the data used to train the model: solubility in each solvent, overlaps between solvents, solvents that behave alike, the effect of RT and molecular weight, and the composition of the training set (original data vs plates).

It reads `data/interim/molecules_unique_updated.xlsx`, built by `scripts/data_prep/add_new_molecules.py`. Figures are saved to `results/figures/eda/data_analysis/`.

In [ ]:
import sys
from pathlib import Path
# Find the project root (the folder that contains src/config.py) and make src/ importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts" / "data_prep"))
from config import MOLECULES_UNIQUE_UPDATED, EXPERIMENTS, FIG_EDA, ensure_dir
from add_new_molecules import read_experiment, experiment_files

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from rdkit import Chem
from rdkit.Chem import Descriptors

FIG_DATA = ensure_dir(FIG_EDA / "data_analysis")
SOLVENTS = ['MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3']
# One fixed colour per solvent (colour-blind safe); solvent names are always written on the charts
COLORS = dict(zip(SOLVENTS, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']))
SOURCE_COLORS = {"original": '#2a78d6', "plates": '#eb6834'}
INK, MUTED, GRID = '#2b2b2b', '#6b6b6b', '#e6e6e3'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': MUTED,
                     'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'font.size': 10, 'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

def save(fig, name):
    fig.savefig(FIG_DATA / name)
    plt.show()

### 1. The dataset

In [ ]:
df = pd.read_excel(MOLECULES_UNIQUE_UPDATED)
df.columns = df.columns.str.strip()
df['CAS'] = df['CAS'].astype(str).str.strip()
df = df[df['solute_smiles'].notna() & df[SOLVENTS].notna().any(axis=1)].reset_index(drop=True)
df['MolWt'] = df['solute_smiles'].map(lambda s: Descriptors.MolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan)

# Where does each molecule come from, and who gave its labels?
plate_records = []
for path in experiment_files():
    recs, _ = read_experiment(path)
    plate_records += [dict(r, plate=path.stem.replace('_output', '')) for r in recs]
plates = pd.DataFrame(plate_records)
plate_of = plates.groupby('cas')['plate'].first() if len(plates) else pd.Series(dtype=str)
df['source'] = np.where(df['CAS'].isin(plate_of.index), 'plates', 'original')
df['plate'] = df['CAS'].map(plate_of).fillna('original data')
df['hplc_column'] = df['Matrix ID Name'].str.replace('matrix ID ', '', regex=False)

print(f"{len(df)} usable molecules")
display(pd.crosstab(df['plate'], df['hplc_column'], margins=True, margins_name='total'))
if len(plates):
    print(f"Plate labels: {plates['from_observation'].sum()} from your observation, "
          f"{(~plates['from_observation']).sum()} from the vision model prediction")

### 2. Probability of being soluble in each solvent

Share of soluble molecules for each solvent: over all molecules, then separately for the original data and for the plates.

In [ ]:
stats = pd.DataFrame({
    'tested': df[SOLVENTS].notna().sum(),
    'soluble': (df[SOLVENTS] == 1).sum(),
    'P(soluble) all': df[SOLVENTS].mean(),
    'P(soluble) original': df[df['source'] == 'original'][SOLVENTS].mean(),
    'P(soluble) plates': df[df['source'] == 'plates'][SOLVENTS].mean(),
}).T
display(stats.style.format(lambda v: f"{v:.0%}" if v <= 1 else f"{v:.0f}", na_rep="-"))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(SOLVENTS)); w = 0.38
for i, src in enumerate(['original', 'plates']):
    vals = df[df['source'] == src][SOLVENTS].mean().values
    n = (df['source'] == src).sum()
    bars = ax.bar(x + (i - 0.5) * w, vals, w - 0.02, color=SOURCE_COLORS[src], label=f"{src} ({n} molecules)")
    for b, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.0%}", ha='center', va='bottom', fontsize=8.5, color=INK)
ax.set_xticks(x, SOLVENTS); ax.set_ylim(0, 1.08); ax.set_ylabel("share of soluble molecules")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Probability of being soluble, per solvent", loc='left', fontweight='bold')
ax.legend(frameon=False, loc='upper left', ncols=2); ax.grid(axis='x', visible=False)
save(fig, "p_soluble_per_solvent.png")

#### Histogram: in how many solvents is a molecule soluble?

In [ ]:
n_sol = (df[SOLVENTS] == 1).sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 3.6))
counts = pd.crosstab(n_sol, df['source']).reindex(range(6), fill_value=0)
bottom = np.zeros(6)
for src in ['original', 'plates']:
    if src in counts:
        ax.bar(counts.index, counts[src], 0.7, bottom=bottom, color=SOURCE_COLORS[src], label=src, edgecolor='white', linewidth=2)
        bottom += counts[src].values
for k, tot in enumerate(bottom):
    ax.text(k, tot + 0.8, f"{int(tot)}", ha='center', fontsize=9)
ax.set_xticks(range(6)); ax.set_xlabel("number of solvents the molecule is soluble in"); ax.set_ylabel("molecules")
ax.set_title("Molecules by number of solvents they dissolve in", loc='left', fontweight='bold')
ax.legend(frameon=False); ax.grid(axis='x', visible=False)
save(fig, "histogram_number_of_solvents.png")
print(f"Soluble nowhere: {(n_sol == 0).sum()} | everywhere: {(n_sol == 5).sum()}")

### 3. Overlaps between solvents

Venn diagrams with 3 solvents (beyond that, a Venn diagram becomes unreadable). Each area gives the number of molecules soluble in those solvents and not in the others of the diagram. The next chart gives, for all 5 solvents at once, the most frequent exact combinations.

In [ ]:
def venn(ax, trio):
    sets = [set(df.index[df[s] == 1]) for s in trio]
    v = venn3(sets, set_labels=trio, set_colors=[COLORS[s] for s in trio], alpha=0.45, ax=ax)
    for t in v.subset_labels:
        if t: t.set_fontsize(9)
    ax.set_title(" / ".join(trio), fontsize=10)

trios = [('MeOH', 'ACN', 'DMSO'), ('DCM', 'CHCl3', 'DMSO'), ('MeOH', 'DCM', 'CHCl3')]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, trio in zip(axes, trios):
    venn(ax, trio)
fig.suptitle(f"Soluble molecules: overlaps ({len(df)} molecules)", x=0.01, ha='left', fontweight='bold')
save(fig, "venn_solvents.png")

In [ ]:
combo = df[SOLVENTS].fillna(0).astype(int).apply(lambda r: " + ".join(s for s in SOLVENTS if r[s]) or "none", axis=1)
top = combo.value_counts().head(12)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(top.index, top.values, color='#2a78d6', height=0.65)
for i, v in enumerate(top.values):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=9)
ax.set_xlabel("molecules"); ax.grid(axis='y', visible=False)
ax.set_title("Most frequent exact solvent combinations", loc='left', fontweight='bold')
save(fig, "solvent_combinations.png")

### 4. Which solvents behave alike?

**Conditional probability**: cell (row A, column B) gives the probability that a molecule is soluble in B **given** that it is soluble in A. If P(CHCl3 | DCM) is close to 100 %, knowing that a molecule dissolves in DCM almost guarantees it dissolves in CHCl3.

**Agreement**: share of molecules with the same result (soluble or insoluble) in both solvents. This is the measure used to **group solvents that behave alike**.

In [ ]:
L = df[SOLVENTS]
cond = pd.DataFrame({b: [L.loc[L[a] == 1, b].mean() for a in SOLVENTS] for b in SOLVENTS}, index=SOLVENTS)
agree = pd.DataFrame({b: [(L[a] == L[b]).mean() for a in SOLVENTS] for b in SOLVENTS}, index=SOLVENTS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, mat, title in [(axes[0], cond, "P(soluble in column | soluble in row)"),
                       (axes[1], agree, "Agreement between solvents (same result)")]:
    im = ax.imshow(mat.values, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(5), SOLVENTS); ax.set_yticks(range(5), SOLVENTS); ax.grid(False)
    for i in range(5):
        for j in range(5):
            v = mat.values[i, j]
            ax.text(j, i, f"{v:.0%}", ha='center', va='center', fontsize=9, color='white' if v > 0.6 else INK)
    ax.set_title(title, fontsize=10, loc='left')
fig.colorbar(im, ax=axes, fraction=0.025, format=plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
save(fig, "solvent_similarity.png")

pairs = sorted(((agree.loc[a, b], a, b) for a, b in itertools.combinations(SOLVENTS, 2)), reverse=True)
print("Most similar solvent pairs (agreement):")
for v, a, b in pairs:
    print(f"  {a:5s} - {b:5s}: {v:.0%}")

### 5. Solubility vs RT and molecular weight

Molecules are sorted into 5 equal-size groups (quintiles) by RT, then by molecular weight. For each group, we compute the share of molecules soluble in each solvent.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for ax, var, label in [(axes[0], 'RT', "RT (min)"), (axes[1], 'MolWt', "molecular weight (g/mol)")]:
    bins = pd.qcut(df[var], 5)
    centers = df.groupby(bins, observed=True)[var].median()
    ends = {}
    for s in SOLVENTS:
        p = df.groupby(bins, observed=True)[s].mean()
        ax.plot(centers.values, p.values, '-o', color=COLORS[s], lw=2, ms=8, mec='white', mew=1.5)
        ends[s] = p.values[-1]
    # solvent name at the end of each line, pushed apart when two names would overlap
    x_text = centers.values[-1] + 0.03 * (centers.values[-1] - centers.values[0])
    last = None
    for s, y in sorted(ends.items(), key=lambda kv: kv[1]):
        y_text = y if last is None else max(y, last + 0.045)
        ax.text(x_text, y_text, s, va='center', fontsize=9, color=INK)
        last = y_text
    ax.set_xlabel(f"{label}, group median"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].set_ylabel("share of soluble molecules"); axes[0].set_ylim(0, 1.05)
fig.suptitle("Solubility vs RT and molecular weight (quintiles)", x=0.01, ha='left', fontweight='bold')
save(fig, "solubility_vs_rt_molwt.png")

### 6. The training set: original data vs plates

Do the plates look like the original data? If their molecules have very different RTs or weights, the model learns in new regions. That is useful, but it must be known to interpret the scores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, var, label in [(axes[0], 'RT', "RT (min)"), (axes[1], 'MolWt', "molecular weight (g/mol)")]:
    bins = np.histogram_bin_edges(df[var].dropna(), bins=15)
    for src in ['original', 'plates']:
        d = df[df['source'] == src][var].dropna()
        if len(d):
            ax.hist(d, bins=bins, histtype='step', lw=2, color=SOURCE_COLORS[src], density=True,
                    label=f"{src} (median {d.median():.1f})")
    ax.set_xlabel(label); ax.set_yticks([]); ax.legend(frameon=False, fontsize=9)
fig.suptitle("RT and molecular weight distributions", x=0.01, ha='left', fontweight='bold')
save(fig, "training_set_distributions.png")

summary = df.groupby('source').agg(molecules=('CAS', 'size'), RT_median=('RT', 'median'), MolWt_median=('MolWt', 'median'),
                                   not_BlueBird_1=('hplc_column', lambda c: (c != 'BlueBird 1').sum()))
display(summary)

## Part B: the model

This part evaluates **Model E** (Arthur's final model, the one in notebooks 01 to 04): one row per molecule x HPLC column x solvent, with RT, molecular weight, the solvent's dielectric constant and Hansen parameters, and the 10 standards of the HPLC column.

The overall score of Model E is in notebook `04_robustness_check`; this part goes further. All results come from the same kind of **cross-validation**: the molecules are split into 5 groups, each group is the test set once, and the whole procedure is repeated with different splits. Every molecule is therefore predicted by a model that has never seen it. A molecule measured on several HPLC columns gets the **mean** of the probabilities predicted on its columns.

This part reads `data/interim/model_e_dataset.xlsx`, built by `scripts/data_prep/add_new_molecules.py`. Figures are saved to `results/figures/model_analysis/`.

### Settings

In [ ]:
N_FOLDS = 5                                     # 5-fold cross-validation
N_REPEATS = 3                                   # number of different splits (more = more stable, but slower)
FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]           # training-set sizes for the learning curve

In [ ]:
from config import MODEL_E_DATASET, MATRIX_IDS_TABLE, FIGURES, ensure_dir
from data_converter.hplc_data_handler import extract_solvent_physics

import xgboost as xgb
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

FIG_MODEL = ensure_dir(FIGURES / "model_analysis")
BLUE, ORANGE, REF = '#2a78d6', '#eb6834', '#8a8a85'
pct = plt.FuncFormatter(lambda v, _: f"{v:.0%}")

def save_model_fig(fig, name):
    fig.savefig(FIG_MODEL / name)
    plt.show()

def make_model_E(weight, seed):
    """Arthur's Model E settings (rt_prediction_continued.ipynb, cell 18)."""
    return xgb.XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=4, subsample=0.8, colsample_bytree=0.5,
                             colsample_bynode=0.9, gamma=.8, min_child_weight=10, reg_lambda=5, reg_alpha=1,
                             scale_pos_weight=weight, random_state=seed, n_jobs=-1)

def first_choice(df):
    """For each molecule: the solvent ranked first and whether that choice was right."""
    keys = [c for c in ['rep'] if c in df.columns] + ['CAS']
    rows = []
    for values, g in df.groupby(keys):
        values = values if isinstance(values, tuple) else (values,)
        best = g.loc[g['p'].idxmax()]
        dmso = g[g['Solvent'] == 'DMSO']
        rows.append(dict(zip(keys, values), choice=best['Solvent'], correct=best['y'] == 1,
                         soluble_somewhere=g['y'].max() == 1,
                         dmso_correct=bool(len(dmso)) and dmso['y'].iloc[0] == 1,
                         dmso_insoluble=bool(len(dmso)) and dmso['y'].iloc[0] == 0))
    return pd.DataFrame(rows)

### 1. Preparing the data

The same features as in notebook 01. Molecules are identified by their CAS.

In [ ]:
_molwt = {}
def molecular_weight(smiles):
    if smiles not in _molwt:
        mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
        _molwt[smiles] = Descriptors.MolWt(mol) if mol else np.nan
    return _molwt[smiles]

physics = {s: extract_solvent_physics(s).values for s in SOLVENTS}   # dielectric, Hansen D, P, H
standards = pd.read_excel(MATRIX_IDS_TABLE).set_index('MatrixID')[[f'M{i}' for i in range(1, 11)]]

# one row per molecule x HPLC column x solvent
e = pd.read_excel(MODEL_E_DATASET)
e['CAS'] = e['CAS '].astype(str).str.strip()
e = e[(e['RT'] != 0) & e['solute_smiles'].notna()]
e['Matrix'] = e['Matrix ID Name'].str.replace('matrix ID ', '', regex=False).str.strip()
e = e.melt(id_vars=['CAS', 'RT', 'Matrix', 'solute_smiles'], value_vars=SOLVENTS,
           var_name='Solvent', value_name='Soluble').dropna(subset=['Soluble'])
e['MolWt'] = e['solute_smiles'].map(molecular_weight)
e = e.dropna(subset=['MolWt']).reset_index(drop=True)
X = np.hstack([e[['RT', 'MolWt']].values,
               np.vstack(e['Solvent'].map(physics).values),
               np.vstack(e['Matrix'].map(lambda m: standards.loc[m].values.astype(float)).values)])
y = e['Soluble'].astype(int).values
molecules = np.array(sorted(e['CAS'].unique()))
print(f"{len(e)} rows, {len(molecules)} molecules")

### 2. Cross-validation

For each split and each training-set size, Model E is trained and then predicts the molecules of the test fold.

This cell takes a few minutes.

In [ ]:
def predict_fold(train_mask, test_mask, seed):
    """Train Model E on train_mask, return one row per test molecule x solvent (mean probability over its columns)."""
    model = make_model_E((y[train_mask] == 0).sum() / (y[train_mask] == 1).sum(), seed).fit(X[train_mask], y[train_mask])
    pred = e.loc[test_mask, ['CAS', 'Solvent']].assign(p=model.predict_proba(X[test_mask])[:, 1], y=y[test_mask])
    return pred.groupby(['CAS', 'Solvent'], as_index=False).agg(p=('p', 'mean'), y=('y', 'first'))

predictions = []
for rep in range(N_REPEATS):
    order = np.random.RandomState(rep).permutation(len(molecules))
    fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[order])
    for k in range(N_FOLDS):
        test_cas = set(fold[fold == k].index)
        train_cas = np.array(sorted(set(molecules) - test_cas))
        for frac in FRACTIONS:
            sub_train = set(np.random.RandomState(100 * rep + k).permutation(train_cas)[:int(round(frac * len(train_cas)))])
            pred = predict_fold(e['CAS'].isin(sub_train).values, e['CAS'].isin(test_cas).values, rep)
            predictions.append(pred.assign(rep=rep, fold=k, fraction=frac, n_train=len(sub_train)))
    print(f"split {rep + 1}/{N_REPEATS} done")

res = pd.concat(predictions, ignore_index=True)
res['pred'] = (res['p'] >= 0.5).astype(int)
full = res[res['fraction'] == 1.0].copy()      # models trained on the whole training set
top1 = first_choice(full)                      # solvent ranked first for each molecule
print(f"Correct solvent choice (top-1): {top1['correct'].mean():.1%}   (always DMSO: {top1['dmso_correct'].mean():.1%})")

### 3. Scores per solvent

Accuracy of Model E for each solvent, compared with the best constant answer for that solvent. A bar below the grey line means that, for this solvent, the model does no better than giving the same answer everywhere.

In [ ]:
per_solvent = full.groupby('Solvent').apply(lambda g: accuracy_score(g['y'], g['pred']), include_groups=False).reindex(SOLVENTS)
ref_solvent = full.groupby('Solvent')['y'].mean().reindex(SOLVENTS)
ref_solvent = np.maximum(ref_solvent, 1 - ref_solvent)    # best constant answer (always soluble or always insoluble)

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(SOLVENTS))
ax.bar(x, per_solvent.values, 0.6, color=BLUE, label='Model E')
for j, v in enumerate(ref_solvent.values):
    ax.hlines(v, j - 0.38, j + 0.38, color=REF, lw=2, linestyles='--', label='constant answer' if j == 0 else None)
ax.set_xticks(x, SOLVENTS); ax.yaxis.set_major_formatter(pct); ax.set_ylim(0, 1); ax.set_ylabel("accuracy")
ax.grid(axis='x', visible=False); ax.legend(frameon=False, ncols=2, loc='upper right')
ax.set_title("Accuracy per solvent", loc='left', fontweight='bold')
save_model_fig(fig, "accuracy_per_solvent.png")
display(pd.DataFrame({'Model E': per_solvent, 'constant answer': ref_solvent}).style.format("{:.0%}"))

#### Confusion matrices

For each solvent: rows are the truth, columns the prediction. The errors to watch are **"insoluble" predicted "soluble"** (top right): the model sends the molecule to a solvent where it will not dissolve.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2), gridspec_kw={"wspace": 0.15})
for ax, s in zip(axes, SOLVENTS):
    gs = full[full['Solvent'] == s]
    cm = confusion_matrix(gs['y'], gs['pred'], labels=[0, 1]) / N_REPEATS
    ax.imshow(cm, cmap='Blues'); ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:.0f}", ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else '#2b2b2b')
    ax.set_xticks([0, 1], ['insol.', 'sol.']); ax.set_yticks([0, 1], ['insol.', 'sol.'] if s == SOLVENTS[0] else [])
    ax.set_title(s); ax.set_xlabel('predicted')
axes[0].set_ylabel('actual')
fig.suptitle("Model E confusion matrices (mean number of molecules per split)", x=0.01, ha='left', fontweight='bold')
save_model_fig(fig, "confusion_matrices_model_e.png")

### 4. Learning curve

The model is retrained on only part of the training molecules (20 %, 40 % ... 100 %), then tested on the same test molecules. If the curves **are still rising** on the right, adding molecules should improve the model. If they are **flat**, more data of the same kind will not help much.

In [ ]:
def scores(g):
    return pd.Series({'top1': first_choice(g)['correct'].mean(), 'bacc': balanced_accuracy_score(g['y'], g['pred'])})

curve = res.groupby(['fraction', 'rep']).apply(scores, include_groups=False).reset_index()
n_train = res.groupby('fraction')['n_train'].mean()
curve_summary = curve.drop(columns='rep').groupby('fraction').agg(['mean', 'std'])
ref_dmso = top1['dmso_correct'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, col, title, ref, ref_name in [(axes[0], 'top1', "Correct solvent choice (top-1)", ref_dmso, "always DMSO"),
                                      (axes[1], 'bacc', "Balanced accuracy", 0.5, "random")]:
    mean, std = curve_summary[(col, 'mean')], curve_summary[(col, 'std')]
    xs = n_train.loc[mean.index].values
    ax.plot(xs, mean.values, '-o', color=BLUE, lw=2, ms=8, mec='white', mew=1.5, label='Model E')
    ax.fill_between(xs, (mean - std).values, (mean + std).values, color=BLUE, alpha=0.12, lw=0)
    ax.axhline(ref, color=REF, ls='--', lw=1.5, label=ref_name)
    ax.set_xlabel("training molecules"); ax.yaxis.set_major_formatter(pct)
    ax.set_title(title, loc='left', fontweight='bold'); ax.legend(frameon=False, fontsize=9)
save_model_fig(fig, "learning_curve.png")
display(curve_summary.style.format("{:.1%}"))

### 5. Probability calibration

Predictions are grouped into bins of announced probability (0-20 %, 20-40 % ...). For each bin, the mean announced probability is compared with the share of molecules **actually** soluble. If the points follow the diagonal, the model is well calibrated: when it announces 80 %, the molecule dissolves 8 times out of 10. This is essential for a future cost function, which will rely on these probabilities.

The **Brier score** summarises calibration: 0 is perfect, 0.25 corresponds to always announcing 50 %.

In [ ]:
full['bin'] = pd.cut(full['p'], np.linspace(0, 1, 6), include_lowest=True)
calib = full.groupby('bin', observed=True).agg(announced_proba=('p', 'mean'), actual_share=('y', 'mean'), predictions=('y', 'size'))
display(calib.style.format({'announced_proba': '{:.0%}', 'actual_share': '{:.0%}', 'predictions': '{:.0f}'}))
brier = np.mean((full['p'] - full['y']) ** 2)
brier_ref = np.mean((full['y'].mean() - full['y']) ** 2)
print(f"Brier score: {brier:.3f}   (reference, always announcing the mean share of soluble: {brier_ref:.3f})")

fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot([0, 1], [0, 1], '--', color=REF, lw=1.5, label='perfect calibration')
ax.plot(calib['announced_proba'], calib['actual_share'], '-o', color=BLUE, lw=2, ms=9, mec='white', mew=1.5, label='Model E')
for _, r in calib.iterrows():
    ax.annotate(f"n={r['predictions']:.0f}", (r['announced_proba'], r['actual_share']), xytext=(6, -12), textcoords='offset points', fontsize=8, color='#6b6b6b')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.xaxis.set_major_formatter(pct); ax.yaxis.set_major_formatter(pct)
ax.set_xlabel("announced probability of solubility"); ax.set_ylabel("share actually soluble")
ax.set_title("Model E calibration", loc='left', fontweight='bold'); ax.legend(frameon=False, loc='upper left')
save_model_fig(fig, "calibration_model_e.png")

### 6. Top-1, top-2, top-3: how many attempts are needed?

For each molecule soluble in at least one solvent, we follow the model's ranking and look at which attempt hits a solvent that works. "Top-2" = a working solvent is among the first 2 suggested.

Reference: **one fixed order** for all molecules, from the most often to the least often soluble solvent.

Top-1 is a bit higher here than the top-1 of notebook 04, because molecules soluble **nowhere** are excluded: for them, no order can find a working solvent.

In [ ]:
fixed_order = list(full.groupby('Solvent')['y'].mean().sort_values(ascending=False).index)
print("Fixed reference order:", " > ".join(fixed_order))

def rank_of_first_hit(g, order=None):
    """Position (1 = first attempt) of the first solvent that works."""
    ranking = g.sort_values('p', ascending=False)['Solvent'].tolist() if order is None else [s for s in order if s in set(g['Solvent'])]
    soluble = set(g.loc[g['y'] == 1, 'Solvent'])
    return next(i + 1 for i, s in enumerate(ranking) if s in soluble)

ranks = []
for (rep, cas), g in full.groupby(['rep', 'CAS']):
    if g['y'].max() == 1:
        ranks.append({'method': 'Model E', 'rank': rank_of_first_hit(g)})
        ranks.append({'method': 'Fixed order', 'rank': rank_of_first_hit(g, fixed_order)})
ranks = pd.DataFrame(ranks)
topk = pd.DataFrame({k: ranks.groupby('method')['rank'].apply(lambda r: (r <= k).mean()) for k in range(1, 6)}).T
topk.index = [f"top-{k}" for k in topk.index]
display(topk[['Model E', 'Fixed order']].style.format("{:.1%}"))
print("Mean number of attempts:", ranks.groupby('method')['rank'].mean().round(2).to_dict())

fig, ax = plt.subplots(figsize=(7, 4))
for m, color in [('Model E', BLUE), ('Fixed order', REF)]:
    ax.plot(range(1, 6), topk[m].values, marker='o', color=color, lw=2, ms=8, mec='white', mew=1.5, label=m, ls='--' if m == 'Fixed order' else '-')
ax.set_xticks(range(1, 6), [f"top-{k}" for k in range(1, 6)]); ax.yaxis.set_major_formatter(pct)
ax.set_ylabel("molecules with a working solvent found"); ax.legend(frameon=False)
ax.set_title("Working solvent among the first k suggested", loc='left', fontweight='bold')
save_model_fig(fig, "top_k.png")

### 7. Feature importance

To measure a feature's importance, it is **randomly shuffled** in the test data (the model can no longer use it) and we look at how much the balanced accuracy drops. The bigger the drop, the more the model relies on that feature. The 4 solvent properties and the 10 column standards are each shuffled together, as one block.

In [ ]:
feature_groups = {'RT': [0], 'Molecular weight': [1], 'Solvent (dielectric + Hansen)': [2, 3, 4, 5], 'HPLC column (10 standards)': list(range(6, 16))}
rng = np.random.RandomState(0)
mol_order = np.random.RandomState(0).permutation(len(molecules))
cv_fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[mol_order])
drops = []
for k in range(N_FOLDS):
    te = e['CAS'].isin(set(cv_fold[cv_fold == k].index)).values
    model = make_model_E((y[~te] == 0).sum() / (y[~te] == 1).sum(), 0).fit(X[~te], y[~te])
    base = balanced_accuracy_score(y[te], model.predict(X[te]))
    for name, cols in feature_groups.items():
        for _ in range(5):
            Xp = X[te].copy()
            Xp[:, cols] = Xp[rng.permutation(len(Xp))][:, cols]
            drops.append({'feature': name, 'drop': base - balanced_accuracy_score(y[te], model.predict(Xp))})
imp = pd.DataFrame(drops).groupby('feature')['drop'].agg(['mean', 'std']).sort_values('mean')
display(imp.style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(imp.index, imp['mean'], xerr=imp['std'], color=BLUE, height=0.55, error_kw={'ecolor': '#6b6b6b', 'lw': 1})
ax.axvline(0, color='#6b6b6b', lw=1); ax.grid(axis='y', visible=False)
ax.set_xlabel("drop in balanced accuracy when the feature is shuffled")
ax.set_title("Model E feature importance", loc='left', fontweight='bold')
save_model_fig(fig, "feature_importance.png")

### 8. Error analysis

The molecule x solvent pairs that Model E gets **wrong in every split**. They are either genuinely atypical molecules or **doubtful labels** worth re-checking, especially those coming from the vision model.

In [ ]:
# origin of each label: original data, or plate (your observation / vision model)
wells = []
for path in experiment_files():
    recs, _ = read_experiment(path)
    wells += [dict(r, plate=path.stem.replace('_output', '')) for r in recs]
wells = pd.DataFrame(wells)
plate_of = wells.groupby('cas')['plate'].first() if len(wells) else pd.Series(dtype=str)
label_origin = {(r['cas'], r['solvent']): ('observation' if r['from_observation'] else 'vision') for _, r in wells.iterrows()}

names = pd.read_excel(MODEL_E_DATASET)
names = names.assign(CAS=names['CAS '].astype(str).str.strip()).drop_duplicates('CAS').set_index('CAS')['Sample Name']
info = e.groupby('CAS').agg(RT_median=('RT', 'median'), MolWt=('MolWt', 'first'))

err = full.assign(wrong=full['pred'] != full['y']).groupby(['CAS', 'Solvent']).agg(error_rate=('wrong', 'mean'), actual=('y', 'first'), proba=('p', 'mean')).reset_index()
err['Molecule'] = err['CAS'].map(names)
err['Source'] = err['CAS'].map(plate_of).fillna("original data")
err['Label'] = [label_origin.get((c, s), "original data") for c, s in zip(err['CAS'], err['Solvent'])]
err = err.join(info, on='CAS')

print("Error rate by label origin:")
display(err.groupby('Label')['error_rate'].agg(['mean', 'size']).rename(columns={'mean': 'error rate', 'size': 'molecule x solvent pairs'}).style.format({'error rate': '{:.0%}'}))

always_wrong = err[err['error_rate'] == 1].assign(gap=lambda x: (x['proba'] - x['actual']).abs()).sort_values('gap', ascending=False)
print(f"{len(always_wrong)} molecule x solvent pairs always wrong (out of {len(err)}). The 20 clearest:")
display(always_wrong.head(20)[['Molecule', 'CAS', 'Solvent', 'actual', 'proba', 'RT_median', 'MolWt', 'Source', 'Label']]
        .rename(columns={'actual': 'actual (1 = soluble)', 'proba': 'predicted proba', 'RT_median': 'RT (median)'})
        .style.format({'predicted proba': '{:.0%}', 'RT (median)': '{:.2f}', 'MolWt': '{:.0f}'}).hide(axis='index'))

### 9. Effect of each plate

Model E is retrained adding the plates **one by one**, in the order of their names. The tested molecules always stay the same (all of them, plates included), with the same splits: only the training set changes. We track the correct solvent choice over all molecules, and over those **insoluble in DMSO**, the case the plates target.

This cell takes one to two minutes.

In [ ]:
plate_list = sorted(plate_of.unique())
variants = [("original data", set())]
for i in range(len(plate_list)):
    variants.append(("+ " + " + ".join(p.replace('MatrixID_', '') for p in plate_list[:i + 1]), set(plate_list[:i + 1])))

effect = []
for name, included in variants:
    excluded = set(plate_of[~plate_of.isin(included)].index)       # molecules of plates not added yet
    for rep in range(N_REPEATS):
        mol_order = np.random.RandomState(rep).permutation(len(molecules))
        cv_fold = pd.Series(np.arange(len(molecules)) % N_FOLDS, index=molecules[mol_order])
        preds = []
        for k in range(N_FOLDS):
            te = e['CAS'].isin(set(cv_fold[cv_fold == k].index)).values
            tr = ~te & ~e['CAS'].isin(excluded).values
            preds.append(predict_fold(tr, te, rep))
        pr = pd.concat(preds)
        t1 = first_choice(pr)
        effect.append({'variant': name, 'rep': rep, 'molecules added': len(plate_of) - len(excluded),
                       'correct choice (all)': t1['correct'].mean(),
                       'correct choice (DMSO-insoluble)': t1.loc[t1['dmso_insoluble'], 'correct'].mean(),
                       'balanced accuracy': balanced_accuracy_score(pr['y'], (pr['p'] >= 0.5).astype(int))})
    print(f"{name}: done")

effect = pd.DataFrame(effect).groupby('variant', sort=False).mean(numeric_only=True).drop(columns='rep')
display(effect.style.format({'molecules added': '{:.0f}', 'correct choice (all)': '{:.1%}',
                             'correct choice (DMSO-insoluble)': '{:.1%}', 'balanced accuracy': '{:.1%}'}))

fig, ax = plt.subplots(figsize=(8, 4))
xs = range(len(effect))
for col, color in [('correct choice (all)', BLUE), ('correct choice (DMSO-insoluble)', ORANGE)]:
    ax.plot(xs, effect[col].values, '-o', color=color, lw=2, ms=8, mec='white', mew=1.5, label=col)
ax.set_xticks(list(xs), effect.index, rotation=15, ha='right'); ax.yaxis.set_major_formatter(pct)
ax.set_ylabel("correct solvent choice"); ax.legend(frameon=False)
ax.set_title("Effect of adding the plates (Model E)", loc='left', fontweight='bold')
save_model_fig(fig, "plate_effect.png")